<a href="https://colab.research.google.com/github/Jasp3r16/thesis_generative_timber/blob/main/23_timber_stock_dataset_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 23_timber_stock_dataset_generation  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-02-24
### Properties script
**Goal:** to generate the timber stock datasets, that the algorithm can use to choose elements for the design

**Inputs:**
*   parameters for properties of the stock
*   search space of geometry

**Outputs:**
*   CSV file with dataset of new stock
*   CSV file with dataset of reused stock

## Mounting Google drive

In [ ]:
import os
import sys
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Definieer je paden (Pas de naam 'Thesis_Project' aan naar jouw mapnaam)
BASE_PATH = '/content/drive/MyDrive/Thesis_TU/'
DATA_PATH = os.path.join(BASE_PATH, 'data_thesis/')
SCRIPT_PATH = os.path.join(BASE_PATH, 'notebooks_thesis/')
OUTPUT_PATH = os.path.join(BASE_PATH, 'research_exports/')

# 3. Voeg de Scripts map toe aan het systeem-pad
# Hierdoor kun je .py bestanden uit die map 'importen'
if SCRIPT_PATH not in sys.path:
    sys.path.append(SCRIPT_PATH)

print(f"✅ Drive mounted. Project directory: {BASE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted. Project directory: /content/drive/MyDrive/Thesis_TU/


# NEW STOCK

## Parameters and inputs

### New stock

In [7]:
# ==========================================
# CEL 1: INPUT TUPLES VOOR CATALOGUS
# ==========================================

# Definieer de dimensies als tuples
# Je kunt deze lijst zo lang of kort maken als je wilt
TUPLE_LENGTHS = (2400, 2700, 3000, 3300, 3600, 4000)
TUPLE_DEPTHS = (100, 150, 175, 200, 225, 250, 300)
TUPLE_WIDTHS = (38, 50, 75, 100)

# Statische Mechanische Parameters (C24 Referentiewaarden)
# Omdat dit een catalogus is, gebruiken we vaste, nominale waarden
MECH_PROPS = {
    'E_modulus_eff': 11000.0, # N/mm2
    'f_mk': 24.0,             # N/mm2
    'Density': 420.0          # kg/m3
}

print("Catalogus parameters succesvol geladen! Ga naar cel 2.")

Catalogus parameters succesvol geladen! Ga naar cel 2.


## Start script

In [13]:
# ==========================================
# CEL 2: CATALOGUS GENERATIE EN EXPORT
# ==========================================
import pandas as pd
import itertools

def generate_timber_catalog():
    data = []

    # itertools.product genereert alle mogelijke combinaties van de 3 tuples
    combinaties = list(itertools.product(TUPLE_LENGTHS, TUPLE_DEPTHS, TUPLE_WIDTHS))

    print(f"Catalogus genereren... Verwacht aantal unieke elementen: {len(combinaties)}")

    for index, (length, depth, width) in enumerate(combinaties):
        data.append({
            'Member_ID': f"NS_{index:05d}",  # CAT-prefix voor Catalogus
            'State': 0,                       # 0 = Nieuw hout
            'Length_Actual': int(length),
            'Depth': float(depth),
            'Widht': float(width),
            'E_modulus_eff': float(MECH_PROPS['E_modulus_eff']),
            'f_mk': float(MECH_PROPS['f_mk']),
            'Density': float(MECH_PROPS['Density'])
        })

    # Omzetten naar DataFrame
    df_catalog = pd.DataFrame(data)

    # Export naar CSV met puntkomma separator
    filename = 'new_timber_catalog.csv'
    df_catalog.to_csv(filename, index=False, sep=';')

    print(f"Succes! Catalogus opgeslagen als '{filename}'.")
    return df_catalog

# Voer de functie uit
df_catalog = generate_timber_catalog()

# Toon de eerste 10 rijen om de systematische opbouw te zien
print("\nPreview van de gestructureerde catalogus:")
display(df_catalog.head(10))

Catalogus genereren... Verwacht aantal unieke elementen: 168
Succes! Catalogus opgeslagen als 'new_timber_catalog.csv'.

Preview van de gestructureerde catalogus:


,Member_ID,State,Length_Actual,Depth,Widht,E_modulus_eff,f_mk,Density
0,NS_00000,0,2400,100.0,38.0,11000.0,24.0,420.0
1,NS_00001,0,2400,100.0,50.0,11000.0,24.0,420.0
2,NS_00002,0,2400,100.0,75.0,11000.0,24.0,420.0
3,NS_00003,0,2400,100.0,100.0,11000.0,24.0,420.0
4,NS_00004,0,2400,150.0,38.0,11000.0,24.0,420.0
5,NS_00005,0,2400,150.0,50.0,11000.0,24.0,420.0
6,NS_00006,0,2400,150.0,75.0,11000.0,24.0,420.0
7,NS_00007,0,2400,150.0,100.0,11000.0,24.0,420.0
8,NS_00008,0,2400,175.0,38.0,11000.0,24.0,420.0
9,NS_00009,0,2400,175.0,50.0,11000.0,24.0,420.0


# REUSED STOCK

## Parameters

In [11]:
import pandas as pd
import numpy as np
import random

# 1. DEFINIEER HET DONOR GEBOUW (PAKHUIS SCENARIO)
donor_batches = [
    {"batch_id": "B01", "count": 2, "orig_width": 180, "orig_depth": 600, "orig_length": 12000},
    {"batch_id": "B02", "count": 10, "orig_width": 75, "orig_depth": 225, "orig_length": 5400},
    {"batch_id": "B03", "count": 4, "orig_width": 200, "orig_depth": 200, "orig_length": 4200}
]

# 2. MECHANISCHE EIGENSCHAPPEN (NEN-EN 338 normen naaldhout)
mechanical_props = {
    'C24': {'e_mod': 11000, 'f_mk': 24, 'density': 420},
    'C18': {'e_mod': 9000,  'f_mk': 18, 'density': 380}
}

# 3. DEFINIEER DE VERWERKINGS LOGICA
def process_element(element_data, global_index):
    # Identificatie: Format RS_0000
    member_id = f"RS_{global_index:04d}"

    # Geometrie: Sloop- en schaafverlies
    cut_loss = random.randint(100, 400)
    length_actual = element_data['orig_length'] - cut_loss
    depth = element_data['orig_depth'] - random.randint(10, 16)
    width = element_data['orig_width'] - random.randint(10, 16)

    # Grading (Kwaliteit) bepalen
    grade = np.random.choice(['C24', 'C18'], p=[0.60, 0.40])

    # Koppel mechanische eigenschappen
    e_modulus_eff = mechanical_props[grade]['e_mod']
    f_mk = mechanical_props[grade]['f_mk']
    density = mechanical_props[grade]['density']

    # LCA Aannames
    transport_dist = random.randint(20, 150)

    return {
        # Identificatie geüpdatet conform jouw matrix
        "Member_ID": member_id,
        "State": 1,          # 1 = Reclaimed (0 zou bijv. Virgin EWP kunnen zijn)

        # Geometrie
        "Length_Actual": length_actual,
        "Depth": depth,
        "Width": width,

        # Mechanisch
        "E_modulus_eff": e_modulus_eff,
        "f_mk": f_mk,
        "Density": density,

        # LCA
        "Embodied Carbon Coëfficiënt": 15.0,
        "Transport_Dist": transport_dist,
        "Emmisiefactor": 0.062,
        "Bewerkingsfactor": 1
    }

# 4. GENEREREN VAN DE DATASET MET GLOBALE TELLER
inventory_list = []
current_id_number = 1

for batch in donor_batches:
    for _ in range(batch['count']):
        processed_element = process_element(batch, current_id_number)
        inventory_list.append(processed_element)
        current_id_number += 1  # Verhoog de teller voor het volgende element

# Maak het DataFrame
df_reused_dataset = pd.DataFrame(inventory_list)

# Resultaat bekijken
print("\nPreview van de gestructureerde catalogus:")
display(df_reused_dataset.head(10))

# 5. EXPORTEREN VAN DE DATASET
# Sla het DataFrame op als een CSV-bestand zonder de index-kolom (rijnummers) mee te nemen
bestandsnaam = 'reclaimed_timber_cost_matrix.csv'
df_cost_matrix.to_csv(bestandsnaam, index=False)

print(f"\nDataset succesvol opgeslagen als '{bestandsnaam}'. De download start nu.")


Preview van de gestructureerde catalogus:


,Member_ID,State,Length_Actual,Depth,Width,E_modulus_eff,f_mk,Density,Embodied Carbon Coëfficiënt,Transport_Dist,Emmisiefactor,Bewerkingsfactor
0,RS_0001,1,11783,585,167,11000,24,420,15.0,150,0.062,1
1,RS_0002,1,11817,590,166,9000,18,380,15.0,111,0.062,1
2,RS_0003,1,5131,211,62,9000,18,380,15.0,134,0.062,1
3,RS_0004,1,5192,210,61,11000,24,420,15.0,38,0.062,1
4,RS_0005,1,5074,210,64,9000,18,380,15.0,80,0.062,1
5,RS_0006,1,5269,212,64,11000,24,420,15.0,102,0.062,1
6,RS_0007,1,5025,214,65,9000,18,380,15.0,109,0.062,1
7,RS_0008,1,5196,210,60,11000,24,420,15.0,127,0.062,1
8,RS_0009,1,5037,209,59,11000,24,420,15.0,90,0.062,1
9,RS_0010,1,5241,209,61,11000,24,420,15.0,116,0.062,1



Dataset succesvol opgeslagen als 'reclaimed_timber_cost_matrix.csv'. De download start nu.


# DOWNLOAD

In [10]:
# Gebruik de ingebouwde Colab-module om het bestand direct te downloaden
from google.colab import files
files.download(bestandsnaam)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>